In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

model_id = 'deepseek-ai/deepseek-moe-16b-chat'
df = pd.read_csv('output.csv')
df = df.query(f'model_id == "{model_id}"')

In [ ]:
# decode time
x = df.query('num_predict_expert_per_layer == 0').query('reorder_experts == False').query('early_preempt == False')['cache_rate']
plt.plot(x, df.query('num_predict_expert_per_layer == 0').query('reorder_experts == False').query('early_preempt == False')['decode_stage_forward_time']/1000, color=(1,0,0,1), label='base')
plt.plot(x, df.query('num_predict_expert_per_layer != 0').query('reorder_experts ==  True').query('early_preempt ==  True')['decode_stage_forward_time']/1000, color=(0,0,1,1), label='ours')

plt.title(model_id)
plt.xlabel('cache rate')
plt.ylabel('decode stage time per token (ms)')
plt.ylim(bottom=0)
plt.grid()
plt.legend()
plt.show()

In [ ]:
# prefill time
x = df.query('num_predict_expert_per_layer == 0').query('reorder_experts == False').query('early_preempt == False')['cache_rate']
plt.plot(x, df.query('num_predict_expert_per_layer == 0').query('reorder_experts == False').query('early_preempt == False')['prefill_stage_forward_time']/1000, color=(1,0,0,1), label='base')
plt.plot(x, df.query('num_predict_expert_per_layer != 0').query('reorder_experts ==  True').query('early_preempt ==  True')['prefill_stage_forward_time']/1000, color=(0,0,1,1), label='ours')

plt.title(model_id)
plt.xlabel('cache rate')
plt.ylabel('prefill stage time per token (ms)')
plt.ylim(bottom=0)
plt.grid()
plt.legend()
plt.show()

In [ ]:
# improvement over baseline(decode)
baseline = df.query('num_predict_expert_per_layer == 0').query('reorder_experts == False').query('early_preempt == False')
x = baseline['cache_rate']
ours = df

plt.plot(x, np.ones_like(x), color=(1,0,0,1), label='base-weak2')
plt.plot(x, np.asarray(baseline['decode_stage_forward_time'])/(ours.query('num_predict_expert_per_layer != 0').query('reorder_experts ==  True').query('early_preempt ==  True')['decode_stage_forward_time']), color=(0,0,1,1), label='ours')

plt.title(model_id)
plt.xlabel('cache rate')
plt.ylabel('improvement to base-weak2')
_, top_ylim = plt.gca().get_ylim()
top_ylim = np.round((top_ylim + 0.1) * 5) / 5
plt.ylim(bottom=0,top=top_ylim)
plt.grid()
plt.legend()
plt.show()